# CardioSafe AI - Notebook 06: QTc measurement and validation

PTB-XL has only 117 long-QT labels, too few to train a QT classifier. Instead we **measure QT directly from the 500 Hz signal** (neurokit2 wavelet delineation, QRS onset to T-wave end, on leads II, V1 and V5), correct it for heart rate (Bazett and Fridericia), and validate it against the QT values that PTB-XL+ provides from independent algorithms (GE 12SL, Glasgow Uni-G, ECGDeli).

**Important framing for the report:** the references are *algorithm outputs*, not manual cardiologist measurements, so we report *agreement with established algorithms*, not clinical accuracy.

Outputs used later by the combined module: a validated QTc measurer, its error limits (so we can say "borderline" instead of over-claiming), and a QTc-prolonged flag.

**Runtime: CPU is enough** (any runtime; a high-RAM CPU is faster). Downloads about 2 GB and takes roughly 20-30 minutes in total.

In [ ]:
!pip -q install wfdb neurokit2 joblib
from google.colab import drive
drive.mount('/content/drive')
import os, sys
WORK = '/content/drive/MyDrive/CardioSafe'
sys.path.insert(0, WORK)
os.environ['PYTHONPATH'] = WORK + ':' + os.environ.get('PYTHONPATH', '')

In [ ]:
%%writefile /content/drive/MyDrive/CardioSafe/cardiosafe_qt.py
"""QT / QTc measurement from 12-lead ECGs (neurokit2 DWT delineation) and agreement statistics."""
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

MEASURE_COLS = ['n_beats', 'rr_ms', 'hr', 'qt_median', 'qt_max', 'qtc_baz_median', 'qtc_baz_max',
                'qtc_fri_median', 'qtc_fri_max', 'n_leads_ok']


def _qt_one_lead(clean, rpeaks, fs):
    """Per-beat QT (ms) = T-offset minus QRS-onset on one lead; returns median over plausible beats, count."""
    import neurokit2 as nk
    _, w = nk.ecg_delineate(clean, rpeaks, sampling_rate=fs, method='dwt', show=False)
    on = np.array(w['ECG_R_Onsets'], dtype=float)
    off = np.array(w['ECG_T_Offsets'], dtype=float)
    n = min(len(on), len(off))
    qt = (off[:n] - on[:n]) / fs * 1000.0
    qt = qt[np.isfinite(qt) & (qt > 250) & (qt < 650)]
    return (float(np.median(qt)), len(qt)) if len(qt) >= 3 else (np.nan, len(qt))


def measure_qt(sig, fs=500, r_lead=1, leads=(1, 6, 10)):
    """sig: (T, 12) in mV. R-peaks from `r_lead` (default II); QT on `leads` (default II, V1... index 6 = V1, 10 = V5).
    Returns dict with MEASURE_COLS; NaN where the measurement is not reliable."""
    import neurokit2 as nk
    out = {k: np.nan for k in MEASURE_COLS}
    try:
        clean_r = nk.ecg_clean(sig[:, r_lead], sampling_rate=fs, method='neurokit')
        _, info = nk.ecg_peaks(clean_r, sampling_rate=fs, method='neurokit')
        rp = np.asarray(info['ECG_R_Peaks'])
        if len(rp) < 4:
            return out
        rr = np.diff(rp) / fs
        rr = rr[(rr > 0.3) & (rr < 2.0)]
        if len(rr) < 3:
            return out
        rr_med = float(np.median(rr))
        out.update(n_beats=len(rp), rr_ms=rr_med * 1000, hr=60 / rr_med)
        qts = []
        for l in leads:
            try:
                clean = clean_r if l == r_lead else nk.ecg_clean(sig[:, l], sampling_rate=fs, method='neurokit')
                q, _ = _qt_one_lead(clean, rp, fs)
                if np.isfinite(q):
                    qts.append(q)
            except Exception:
                continue
        out['n_leads_ok'] = len(qts)
        if qts:
            for tag, q in (('median', float(np.median(qts))), ('max', float(np.max(qts)))):
                out[f'qt_{tag}'] = q
                out[f'qtc_baz_{tag}'] = q / np.sqrt(rr_med)
                out[f'qtc_fri_{tag}'] = q / np.cbrt(rr_med)
    except Exception:
        pass
    return out


def measure_record(path, fs_expected=500, **kw):
    """Worker for parallel use: read a WFDB record (full path without extension) and measure."""
    import wfdb
    try:
        sig, fields = wfdb.rdsamp(path)
        return measure_qt(sig.astype(np.float64), fs=int(fields['fs']), **kw)
    except Exception:
        return {k: np.nan for k in MEASURE_COLS}


# ------------------------------------------------------------- agreement
def agreement(ours, ref):
    """Bland-Altman style statistics for paired measurements (ms)."""
    m = np.isfinite(ours) & np.isfinite(ref)
    a, b = np.asarray(ours)[m], np.asarray(ref)[m]
    d = a - b
    sd = d.std(ddof=1)
    return {'n': int(m.sum()), 'bias_ms': float(d.mean()), 'MAE_ms': float(np.abs(d).mean()),
            'SD_diff_ms': float(sd), 'LoA_low': float(d.mean() - 1.96 * sd), 'LoA_high': float(d.mean() + 1.96 * sd),
            'pearson_r': float(np.corrcoef(a, b)[0, 1]),
            'within_20ms': float((np.abs(d) <= 20).mean()), 'within_30ms': float((np.abs(d) <= 30).mean()),
            'within_40ms': float((np.abs(d) <= 40).mean())}


def prolonged_threshold(sex, male_ms=450, female_ms=460):
    """Commonly used sex-specific QTc limits. PTB-XL sex coding: 0 = male, 1 = female."""
    return np.where(np.asarray(sex) == 0, male_ms, female_ms).astype(float)


def classification_stats(flag_pred, flag_true):
    p, t = np.asarray(flag_pred, bool), np.asarray(flag_true, bool)
    tp, fp, fn, tn = (p & t).sum(), (p & ~t).sum(), (~p & t).sum(), (~p & ~t).sum()
    return {'sensitivity': tp / max(tp + fn, 1), 'specificity': tn / max(tn + fp, 1), 'PPV': tp / max(tp + fp, 1),
            'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn)}


## Data: PTB-XL (500 Hz records) and PTB-XL+ feature tables

In [ ]:
%%bash
cd /content
if [ ! -d ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3 ]; then
  wget -q --show-progress -O ptbxl.zip https://physionet.org/static/published-projects/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip
  unzip -q ptbxl.zip
fi
mkdir -p ptbxlplus && cd ptbxlplus
for f in feature_description.csv 12sl_features.csv unig_features.csv ecgdeli_features.csv; do
  [ -s $f ] || wget -q https://physionet.org/files/ptb-xl-plus/1.0.1/features/$f
done
ls -la /content/ptbxlplus

## Choose the validation set
All 117 long-QT-labelled ECGs plus a random sample of other ECGs (seeded). QT measurement involves no model training, so every fold may be used.

In [ ]:
import glob, ast, time, numpy as np, pandas as pd, matplotlib.pyplot as plt
import cardiosafe_qt as Q
from tqdm.auto import tqdm
from joblib import Parallel, delayed
N_RANDOM = 4000

ROOT = glob.glob('/content/ptb-xl-*/')[0]
db = pd.read_csv(ROOT + 'ptbxl_database.csv', index_col='ecg_id')
db['scp_codes'] = db['scp_codes'].apply(ast.literal_eval)
db['lngqt'] = db['scp_codes'].apply(lambda d: 'LNGQT' in d)
rng = np.random.default_rng(0)
pos, rest = db.index[db.lngqt].values, db.index[~db.lngqt].values
sel = np.concatenate([pos, rng.choice(rest, min(N_RANDOM, len(rest)), replace=False)])
sub = db.loc[sel]
print('records:', len(sel), '| long-QT labelled:', len(pos))

## Measure QT/QTc (resumable; result cached to Drive)

In [ ]:
cache = WORK + '/qt_measurements.csv'
if os.path.exists(cache):
    meas = pd.read_csv(cache, index_col='ecg_id'); print('loaded cached measurements', meas.shape)
else:
    paths = [ROOT + f for f in sub['filename_hr']]
    parts, CH, t0 = [], 500, time.time()
    for i in tqdm(range(0, len(sel), CH)):
        r = Parallel(n_jobs=-1, backend='multiprocessing')(delayed(Q.measure_record)(p) for p in paths[i:i + CH])
        parts.append(pd.DataFrame(r, index=sel[i:i + CH]))
    meas = pd.concat(parts); meas.index.name = 'ecg_id'; meas.to_csv(cache)
    print(f'{time.time() - t0:.0f}s')
ok = meas['qt_median'].notna()
print(f'QT measured for {ok.mean() * 100:.1f}% of ECGs | median HR {meas.hr.median():.0f} bpm | median QT {meas.qt_median.median():.0f} ms')
display(meas.describe().T[['count', 'mean', 'std', 'min', 'max']].round(1))

## Reference measurements from PTB-XL+
The next cell discovers the column names in the three feature files (names differ by algorithm) and prints which ones it matched. **If a reference shows no match, send me the printed column list.**

In [ ]:
PLUS = '/content/ptbxlplus'
desc = pd.read_csv(PLUS + '/feature_description.csv')
print('feature_description columns:', desc.columns.tolist())

def load_feat(name):
    df = pd.read_csv(f'{PLUS}/{name}_features.csv')
    idc = [c for c in df.columns if c.lower() in ('ecg_id', 'ecgid', 'id')]
    return df.set_index(idc[0] if idc else df.columns[0])

CAND = {'qt':      ['QT_Int_Global', 'Int_QT_Interval', 'QT_Interval', 'QTi_max'],
        'qtc_baz': ['QT_IntBazett_Global', 'QTc_Bazett'],
        'qtc_fri': ['QT_IntFridericia_Global', 'QTc_Fridericia', 'QTcFredericia'],
        'rr':      ['RR_Mean_Global', 'GM_Average_RR_Sinus', 'AvgRRInterval', 'RRi_max'],
        'hr':      ['HR__Global', 'HR_Ventr_Global', 'VentRate', 'GM_Display_Heart_Rate']}
def find(df, names):
    low = {c.lower(): c for c in df.columns}
    return next((low[n.lower()] for n in names if n.lower() in low), None)

REF = {}
for alg in ['12sl', 'unig', 'ecgdeli']:
    df = load_feat(alg)
    cols = {k: find(df, v) for k, v in CAND.items()}
    print(f'{alg:8s} rows={len(df)} matched:', cols)
    if not any(cols.values()):
        print('   NO MATCH. First 40 columns:', df.columns[:40].tolist())
        print('   QT-like columns:', [c for c in df.columns if 'qt' in c.lower()][:20])
        continue
    REF[alg] = pd.DataFrame({k: df[c].reindex(sel).astype(float) for k, c in cols.items() if c is not None}, index=sel)
assert REF, 'No reference columns found; send me the printed column lists.'

## Agreement: our measurement vs each reference algorithm
`bias` = ours minus reference. `LoA` = 95% limits of agreement (bias +/- 1.96 SD). Compare QT with QT, Bazett with Bazett and Fridericia with Fridericia.

In [ ]:
PAIRS = {'qt_median': 'qt', 'qt_max': 'qt', 'qtc_baz_median': 'qtc_baz', 'qtc_baz_max': 'qtc_baz',
         'qtc_fri_median': 'qtc_fri', 'qtc_fri_max': 'qtc_fri', 'rr_ms': 'rr'}
rows = []
for alg, ref in REF.items():
    for ours, key in PAIRS.items():
        if key in ref.columns:
            rows.append({'reference': alg, 'ours': ours, **Q.agreement(meas.loc[sel, ours].values, ref[key].values)})
agree = pd.DataFrame(rows)
agree.to_csv(WORK + '/results_06_qt_agreement.csv', index=False)
display(agree.round(3).sort_values(['reference', 'ours']))

In [ ]:
# choose the best QTc variant against the first available reference (lowest MAE among QTc rows)
ref_name = list(REF.keys())[0]            # '12sl' if available
cand = agree[(agree.reference == ref_name) & agree.ours.str.startswith('qtc')]
best = cand.sort_values('MAE_ms').iloc[0]
ours_col, key = best['ours'], PAIRS[best['ours']]
print(f'Best variant vs {ref_name}: {ours_col} (MAE {best.MAE_ms:.1f} ms, bias {best.bias_ms:.1f} ms, r = {best.pearson_r:.3f})')
a = meas.loc[sel, ours_col].values; b = REF[ref_name][key].values; m = np.isfinite(a) & np.isfinite(b)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
ax[0].scatter(b[m], a[m], s=4, alpha=.3); lim = [300, 650]; ax[0].plot(lim, lim, 'r--'); ax[0].set_xlim(lim); ax[0].set_ylim(lim)
ax[0].set_xlabel(f'{ref_name} {key} (ms)'); ax[0].set_ylabel(f'our {ours_col} (ms)'); ax[0].set_title('Agreement')
mean_, diff = (a[m] + b[m]) / 2, a[m] - b[m]
ax[1].scatter(mean_, diff, s=4, alpha=.3); ax[1].axhline(diff.mean(), color='k'); ax[1].axhline(diff.mean() + 1.96 * diff.std(), color='r', ls='--'); ax[1].axhline(diff.mean() - 1.96 * diff.std(), color='r', ls='--')
ax[1].set_xlabel('mean of the two (ms)'); ax[1].set_ylabel('ours - reference (ms)'); ax[1].set_title('Bland-Altman')
plt.tight_layout(); plt.savefig(WORK + '/fig_qtc_agreement.png', dpi=150); plt.show()

## Bias correction (cross-validated)
Our measurement may be systematically offset from the reference algorithm. A simple linear correction is fitted on one random half and **evaluated on the other half**, so the reported improvement is not optimistic.

In [ ]:
from sklearn.linear_model import LinearRegression
idx = np.where(m)[0]; r2 = np.random.default_rng(1); r2.shuffle(idx); h = len(idx) // 2
A, B = a[idx], b[idx]
lr = LinearRegression().fit(A[:h, None], B[:h]); pred = lr.predict(A[h:, None])
before, after = Q.agreement(A[h:], B[h:]), Q.agreement(pred, B[h:])
print(pd.DataFrame({'before correction': before, 'after correction': after}).round(3).T[['n', 'bias_ms', 'MAE_ms', 'SD_diff_ms', 'LoA_low', 'LoA_high', 'within_20ms', 'within_30ms']])
lr = LinearRegression().fit(A[:, None], B)                    # final fit on everything, for the demo
sd_resid = float(np.std(B - lr.predict(A[:, None]), ddof=1))
print(f'final correction: QTc_corrected = {lr.coef_[0]:.3f} * ours + {lr.intercept_:.1f} | residual SD {sd_resid:.1f} ms')

## Does the QTc flag agree with the references and with PTB-XL's long-QT label?
Prolonged = QTc at or above 450 ms (male) / 460 ms (female), commonly used limits. The validation set is **enriched** with long-QT ECGs (all 117 plus a random sample), so sensitivity and specificity are meaningful, but predictive values would be inflated and are not reported.

In [ ]:
from sklearn.metrics import roc_auc_score
sex = sub['sex'].values; thr = Q.prolonged_threshold(sex)
corr_all = lr.predict(np.nan_to_num(meas.loc[sel, ours_col].values, nan=0)[:, None]); corr_all[~np.isfinite(meas.loc[sel, ours_col].values)] = np.nan
ours_corr = pd.Series(corr_all, index=sel)
ref_q = pd.Series(REF[ref_name][key].values, index=sel)
lab = sub['lngqt'].values

rows = []
valid = np.isfinite(ours_corr.values) & np.isfinite(ref_q.values)
rows.append({'comparison': f'our QTc flag vs {ref_name} flag', **Q.classification_stats((ours_corr.values >= thr)[valid], (ref_q.values >= thr)[valid])})
for name, s in [('our QTc (corrected)', ours_corr), (f'{ref_name} QTc', ref_q)]:
    v = np.isfinite(s.values)
    st = Q.classification_stats((s.values >= thr)[v], lab[v])
    st.update(AUROC_vs_LNGQT=roc_auc_score(lab[v], s.values[v])); rows.append({'comparison': f'{name} vs PTB-XL LNGQT label', **st})
res = pd.DataFrame(rows).set_index('comparison').drop(columns=['PPV']); display(res.round(3))
res.to_csv(WORK + '/results_06_qtc_flag.csv')
print('Sensitivity/specificity vs the LNGQT label are computed on an enriched set; compare our row with the reference-algorithm row rather than reading absolute values.')

## Save the QTc configuration for the combined module

In [ ]:
import json
loa = 1.96 * sd_resid
cfg = {'variant': ours_col, 'reference': ref_name, 'slope': float(lr.coef_[0]), 'intercept': float(lr.intercept_),
       'residual_sd_ms': sd_resid, 'borderline_band_ms': float(loa), 'male_threshold_ms': 450, 'female_threshold_ms': 460,
       'leads_used': ['II', 'V1', 'V5'], 'sampling_rate_hz': 500}
json.dump(cfg, open(WORK + '/qtc_config.json', 'w'), indent=1); print(cfg)
bord = np.isfinite(ours_corr.values) & (np.abs(ours_corr.values - thr) <= loa)
print(f'ECGs within the +/-{loa:.0f} ms borderline band around the threshold: {bord.mean() * 100:.1f}%')

## Next
Send me: the QT-measured percentage line, the descriptive table, the "matched" lines, the agreement table, the cross-validated correction table and the flag-agreement table (plus `fig_qtc_agreement.png` if you can).
Then notebook 07: the drug knowledge base (openFDA label text + DDInter + optional CredibleMeds) and the prescription pipeline.